<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day21_practice2_RAG_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# RAG (Retrieval-Augmented Generation, 검색 증강 생성) 파이프라인
# 문서 청킹 - 긴 규정집을 검색 단위로 자르기
# 검색 결과를 프롬프트에 '증거'로 주입

In [2]:
import torch
import torch.nn.functional as F
import requests, os, re

In [3]:
# 1) 사전 만들기: 글자 2조각 → 번호, n-gram 임베딩
def make_vocab(texts, n=2):
  grams = set()
  for t in texts:
    t = re.sub(r"\s+", "", t) # \s: 공백 제거
    grams |= {t[i:i+n] for i in range(len(t) - n + 1)} # |=: 합집합, 조각 시작 위치, 모든 2글자 조각을 모은다.
  return {g: i for i, g in enumerate(sorted(grams))}

# 2) 문장 → 벡터 변환
def embed(text, vocab, n=2):
  v = torch.zeros(len(vocab))
  t = re.sub(r"\s+", "", text) # 공백이 없는 문장
  for i in range(len(t) - n + 1): # 2글자씩
    g = t[i:i+n]
    if g in vocab:
      v[vocab[g]] += 1 # 단어 등장 횟수 증가
  return v

# API 호출 - openAI 호환 api 형식으로 코드 작성
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"
MODEL = "gemini-flash-lite-latest" # Google이 업데이트하는 Flash-Lite 라인업의 가장 최신 버전으로 자동 연결

# API 키 - Colab Secrets에서 가져오기
def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

API_KEY = _get_api_key()

def call_llm(messages, temperature=0.2):
  if not API_KEY:
    return None
  try:
    r = requests.post(f"{BASE_URL}/chat/completions",
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"model":MODEL, "messages": messages, "temperature": temperature},
                      timeout=30)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]
  except Exception:
    return None

In [5]:
# 셀 1. 청킹 - 규정집은 통째로 못 넣는다. 문단 단위로 잘라(청크) '필요한 조각만' 검색해 넣는다

규정집 = """제1조(긴급 보수) 심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다. 배정 결과는 신고자에게 통보한다.

제2조(정기 처리) 심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.

제3조(맨홀) 맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다. 이관 후에도 처리 현황을 추적한다.

제4조(낙하물) 낙하물 신고는 2차 사고 예방을 위해 즉시 현장 출동을 원칙으로 한다.

제5조(야간 접수) 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다. 단, 심각도 5는 야간에도 즉시 출동한다.

제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다."""

chunks = [c.strip() for c in 규정집.split("\n\n")] # 문단 기준으로 자름
print(f"청킹: 규정집 → {len(chunks)}개 조각 (문단 단위)")
vocab = make_vocab(chunks)
chunk_vectors = torch.stack([embed(c, vocab) for c in chunks])

청킹: 규정집 → 6개 조각 (문단 단위)


In [7]:
print(chunks)

['제1조(긴급 보수) 심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다. 배정 결과는 신고자에게 통보한다.', '제2조(정기 처리) 심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.', '제3조(맨홀) 맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다. 이관 후에도 처리 현황을 추적한다.', '제4조(낙하물) 낙하물 신고는 2차 사고 예방을 위해 즉시 현장 출동을 원칙으로 한다.', '제5조(야간 접수) 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다. 단, 심각도 5는 야간에도 즉시 출동한다.', '제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.']


In [8]:
print(chunk_vectors.shape)

torch.Size([6, 216])


In [21]:
# 셀 2. RAG의 3단계 : 검색 → 증거 주입 → 생성
# 1) 검색(Retrieve) : 질문과 비슷한 조각 k개 찾기
def retrieve(query, k=2):
  q = embed(query, vocab) # 임베딩 → 벡터화
  sims = F.cosine_similarity(q.unsqueeze(0), chunk_vectors) # 코사인 유사도, (157,)→(1,157), 질문 1개 vs 문서 6개를 한번에 유사도 검사
  top = sims.argsort(descending=True)[:k] # 내림차순으로 정렬 상위 k개 인덱스 번호
  return [(chunks[i], sims[i].item()) for i in top] # 상위 k개 검색 결과

# 2) 프롬프트 템플릿
RAG_PROMPT = """너는 도로 관리 규정 안내 담당자다.
아래 [규정]에 근거해서만 답하라. 규정에 없는 내용은 '규정에 없음'이라고 답하라.

[규정]
{context}

[질문] {question}
[답변] 근거 조항을 인용하며 두 문장 이내로."""

# 3) RAG 전체 흐름: 검색 → 주입 → 생성
def rag_answer(question):
  evidence = retrieve(question, k=2) # 1. 검색 결과
  context = "\n".join(str(e) for e in evidence) # Top K 검색 결과를 한 문장으로
  # context = "\n".join(evidence) # Top K 검색 결과를 한 문장으로
  prompt = RAG_PROMPT.format(context = context, question = question) # 질문과 검색 결과를 프롬프트에 주입, str.format() 값을 채워서 새 문자열 반환
  answer = call_llm([{"role": "user", "content": prompt}])
  return answer, evidence

# 4) 실행
question = "밤에 신고하면 언제 처리 시작돼요?"
answer, evidence = rag_answer(question)
print(f"질문: {question}")
print("검색된 증거:")
for e in evidence: print(f" ·{e}...")
print(f"답변: {answer}")

질문: 밤에 신고하면 언제 처리 시작돼요?
검색된 증거:
 ·('제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.', 0.267261266708374)...
 ·('제5조(야간 접수) 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작한다. 단, 심각도 5는 야간에도 즉시 출동한다.', 0.20412415266036987)...
답변: 제5조(야간 접수)에 따라, 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작합니다. 단, 심각도 5는 야간에도 즉시 출동합니다.


In [22]:
# 셀 3. 대조 실험 - LLM 단독 vs RAG
def llm_solo(question):
  prompt = f"""너는 도로 관리 규정 안내 담당자다. 다음 질문에 두 문장 이내로 답하라.
            [질문] {question}"""
  ans = call_llm([{"role": "user", "content": prompt}])
  return ans or "(LLM 응답 없음 : 키를 확인하세요)"

def llm_rag(question):
  ans, evidence = rag_answer(question)
  return (ans or "(LLM 응답 없음 : 키를 확인하세요)"), evidence

질문들 = [
    "밤에 신고하면 언제 처리 시작돼요?",
    "신고하면 보상금은 얼마나 받나요?",
]
for q in 질문들:
  print(f"질문: {q}\n")
  solo = llm_solo(q)
  print(f"LLM 단독 (규정 없음) 답변:\n {solo}\n")

  rag, evidence = llm_rag(q)
  print(f"RAG (규정 주입) 답변:\n {rag}")
  print(f"[검색된 증거] {evidence[0][:45]}...\n" if evidence else "")

질문: 밤에 신고하면 언제 처리 시작돼요?

LLM 단독 (규정 없음) 답변:
 야간 시간대에 접수된 신고는 익일 오전 근무 시작과 함께 순차적으로 처리가 시작됩니다. 긴급을 요하는 사안의 경우, 야간 당직반을 통해 즉시 현장 조치될 수 있습니다.

RAG (규정 주입) 답변:
 제5조(야간 접수)에 따라, 야간(22시~06시) 접수 신고는 익일 오전 담당자 배정 후 처리를 시작합니다. 단, 심각도 5는 야간에도 즉시 출동합니다.
[검색된 증거] ('제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.', 0.267261266708374)...

질문: 신고하면 보상금은 얼마나 받나요?

LLM 단독 (규정 없음) 답변:
 도로 파손 등 불법 행위를 신고하셔도 관련 규정에 따른 별도의 포상금이나 보상금은 지급되지 않습니다. 다만, 안전한 도로 환경 조성을 위한 공익 신고에 적극적인 참여를 부탁드립니다.

RAG (규정 주입) 답변:
 규정에 없음.
[검색된 증거] ('제6조(완료 통보) 처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.', 0.30860671401023865)...

